# Union de las bases datos y ajustes

En archivo se construira la serie de tiempo de la producción agricola de Colombia, 
usando los datos del Banco de la República, y se construiran las variables dummy para cada reforma agraria.
Cada dataset tiene bases para los datos diferentes, es necario unificarlos para poder hacer un buen analisis
comparativo, ademas se debe ajustar la inflación y aplicar una metodologia de desestacionalizacion para poder hacer
un analisis econometrico de series de tiempo.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# para leer los datos en xlsx
import seaborn as sns

In [4]:
# vamos a leer todas las bases de datos

for i in range(3, 9):
    # Creamos la variable globalmente con el nombre dinámico
    globals()[f'df{i}'] = pd.read_excel(f'File ({i}).xlsx', sheet_name='Series de datos', skiprows=1)
    print(f'Variable df{i} creada exitosamente.')



Variable df3 creada exitosamente.
Variable df4 creada exitosamente.
Variable df5 creada exitosamente.
Variable df6 creada exitosamente.
Variable df7 creada exitosamente.
Variable df8 creada exitosamente.


Ya teniendo todas las bases cargadas vamos a limpiar las filas de NA y seleccionar las filas que se llame 1.01.01. Agricultura, ganadería, caza, silvicultura y pesca y 1.01. Agropecuario

In [5]:
# Creamos una lista con los sectores de interés
sectores_interes = ['1.01.01. Agropecuario, silvicultura, caza y pesca', '1.01. Agropecuario', '1.01.01. Agricultura, silvicultura, caza y pesca'
, '1.01.01. Agricultura, ganadería, caza, silvicultura\ny pesca' ]

for i in range(3, 9):
    # 1. Traemos el dataframe original usando su nombre de variable
    df_actual = globals()[f'df{i}']
    
    # 2. Limpiamos y filtramos el dataframe
    df_actual.dropna(inplace=True)
    filtro_agri = df_actual[df_actual['Unnamed: 0'].isin(sectores_interes)].copy()
    
    # 3. Creamos la nueva variable 
    globals()[f'df{i}_agri'] = filtro_agri
    
    print(f'Creada exitosamente la variable: df{i}_agri con {len(filtro_agri)} filas.')

Creada exitosamente la variable: df3_agri con 1 filas.
Creada exitosamente la variable: df4_agri con 1 filas.
Creada exitosamente la variable: df5_agri con 1 filas.
Creada exitosamente la variable: df6_agri con 1 filas.
Creada exitosamente la variable: df7_agri con 1 filas.
Creada exitosamente la variable: df8_agri con 1 filas.


Ya con las bases donde seleccionamos la agricultura ahora vamos a cambiar los formato de ancho a largo, teniendo en cuenta que las columnas son fechas

In [6]:
sectores_interes = [
    '1.01.01. Agropecuario, silvicultura, caza y pesca', 
    '1.01. Agropecuario', 
    '1.01.01. Agricultura, silvicultura, caza y pesca',
    '1.01.01. Agricultura, ganadería, caza, silvicultura\ny pesca'
]

for i in range(3, 9):
    # 1. Traer el DF original
    df_actual = globals()[f'df{i}'].copy()
    
    # 2. Limpiar nulos
    df_actual.dropna(inplace=True)
    
    # 3. Filtrar usando tu lista de nombres variables
    df_filtrado = df_actual[df_actual['Unnamed: 0'].isin(sectores_interes)].copy()
    
    # 4. HOMOLOGACIÓN: Forzamos a que todos se llamen igual
    # Así, sin importar cómo venía en el Excel, ahora será estándar
    df_filtrado['Unnamed: 0'] = 'Sector Agropecuario'
    
    # 5. Transformar a formato largo (Melt)
    df_long = pd.melt(
        df_filtrado, 
        id_vars=['Unnamed: 0'], 
        var_name='Fecha', 
        value_name='Valor'
    )
    
    # 6. Limpieza final de la columna Fecha y Renombrar
    df_long.rename(columns={'Unnamed: 0': 'Sector'}, inplace=True)
    df_long['Fecha'] = pd.to_datetime(df_long['Fecha'], dayfirst=True, errors='coerce')
    
    # 7. Guardar en variable global
    globals()[f'df{i}_long'] = df_long
    
    print(f'df{i}_long creado y estandarizado con éxito.')

df3_long creado y estandarizado con éxito.
df4_long creado y estandarizado con éxito.
df5_long creado y estandarizado con éxito.
df6_long creado y estandarizado con éxito.
df7_long creado y estandarizado con éxito.
df8_long creado y estandarizado con éxito.


Con los df creados vamos mirar los años en los cuales inicia y finaliza cada df

In [7]:
for i in range(3, 9):
    df_actual = globals()[f'df{i}_long']
    print(f'Para df{i}_long: Fecha mínima = {df_actual["Fecha"].min()} | Fecha máxima = {df_actual["Fecha"].max()}')

Para df3_long: Fecha mínima = 1925-12-31 00:00:00 | Fecha máxima = 1949-12-31 00:00:00
Para df4_long: Fecha mínima = 1950-12-31 00:00:00 | Fecha máxima = 1970-12-31 00:00:00
Para df5_long: Fecha mínima = 1970-12-31 00:00:00 | Fecha máxima = 1996-12-31 00:00:00
Para df6_long: Fecha mínima = 1994-03-31 00:00:00 | Fecha máxima = 2007-12-31 00:00:00
Para df7_long: Fecha mínima = 2000-03-31 00:00:00 | Fecha máxima = 2017-12-31 00:00:00
Para df8_long: Fecha mínima = 2005-03-31 00:00:00 | Fecha máxima = 2025-12-31 00:00:00


Vemos como del df6 al df8 las series son trimestrales, entonces vamos a pasar estos dos series a anuales para poder hacer un analisis homogeneo de estos 100 años

In [8]:
# Lista de los dataframes que son trimestrales
dfs_trimestrales = [6, 7, 8]

for i in dfs_trimestrales:
    df_temp = globals()[f'df{i}_long'].copy()
    
    # 1. Ponemos la fecha como índice (necesario para resample)
    df_temp.set_index('Fecha', inplace=True)
    
    # 2. Agrupamos por año ('YE' o 'Y') y sumamos los valores
    # Usamos .resample('YE').sum() para sumar los 4 trimestres
    df_anual = df_temp.resample('YE').sum(numeric_only=True).reset_index()
    
    # 3. Re-agregamos la columna de Sector (que se pierde al sumar)
    df_anual['Sector'] = 'Sector Agropecuario'
    
    # 4. Guardamos la nueva versión anual
    globals()[f'df{i}_anual'] = df_anual
    
    print(f'df{i} convertido a anual. Periodos: {len(df_anual)}')

df6 convertido a anual. Periodos: 14
df7 convertido a anual. Periodos: 18
df8 convertido a anual. Periodos: 21


# ANTES DE LA UNION
vamos a trasformar todas las series por medio de un proceso de empalme de cada una de ellas con las siguiente ecuación

$$
Y_{t-1} = \frac{Y_t}{1+G_t}
$$

Donde:
$Y_t$ es es el nivel de la serie nueva
$Y_i$ es es el nivel de la serie vieja
$G_t$ crecimiento de la serie antigua

Entonces recordamos en cual base esta cada uno de los df

df_3 = base 1950
df_4 = base 1958
df_5 = base 1975
df_6 = base 1994
df_7 = base 2005
df_8 = base 2015

Vamos a pasar todas las series a base de 2015

In [11]:
# --- 1. PROCESAMIENTO INICIAL ---
sectores_interes = [
    '1.01.01. Agropecuario, silvicultura, caza y pesca', 
    '1.01. Agropecuario', 
    '1.01.01. Agricultura, silvicultura, caza y pesca',
    '1.01.01. Agricultura, ganadería, caza, silvicultura\ny pesca'
]

for i in range(3, 9):
    df = pd.read_excel(f'File ({i}).xlsx', sheet_name='Series de datos', skiprows=1)
    df.dropna(subset=['Unnamed: 0'], inplace=True)
    df_f = df[df['Unnamed: 0'].isin(sectores_interes)].copy()
    df_f['Unnamed: 0'] = 'Sector Agropecuario'
    
    # Melt y Fechas
    df_l = pd.melt(df_f, id_vars=['Unnamed: 0'], var_name='Fecha', value_name='Valor')
    df_l.rename(columns={'Unnamed: 0': 'Sector'}, inplace=True)
    df_l['Fecha'] = pd.to_datetime(df_l['Fecha'], dayfirst=True, errors='coerce')
    df_l['Año'] = df_l['Fecha'].dt.year
    
    if i in [3, 4, 5]:
        globals()[f'df{i}_long'] = df_l.sort_values('Año')
    else:
        # Anualización para 6, 7 y 8 (Suma de trimestres)
        df_a = df_l.resample('YE', on='Fecha').sum(numeric_only=True).reset_index()
        df_a['Año'] = df_a['Fecha'].dt.year
        df_a['Sector'] = 'Sector Agropecuario'
        globals()[f'df{i}_anual'] = df_a.sort_values('Año')

# --- 2. ESTRATEGIA DE EMPALME EN CASCADA ---
series_ajustadas = {8: df8_anual.copy()}
nombres_map = {7: 'df7_anual', 6: 'df6_anual', 5: 'df5_long', 4: 'df4_long', 3: 'df3_long'}

# Empalme de 7 a 4 (Donde hay años comunes directos)
for i in range(7, 3, -1):
    df_ref = series_ajustadas[i+1]
    df_viej = globals()[nombres_map[i]].copy()
    
    comunes = set(df_ref['Año']) & set(df_viej['Año'])
    anio_pivote = max(comunes)
    
    factor = df_ref.loc[df_ref['Año'] == anio_pivote, 'Valor'].values[0] / \
             df_viej.loc[df_viej['Año'] == anio_pivote, 'Valor'].values[0]
    
    df_viej['Valor'] = df_viej['Valor'] * factor
    series_ajustadas[i] = df_viej

# --- 3. SOLUCIÓN DEFINITIVA AL EMPALME DF3 -> DF4 (Sin traslape) ---
# --- 3. EMPALME DF3 -> DF4 USANDO DATOS MANUALES (PIB AGREGADO) ---

df_ref_4 = series_ajustadas[4].sort_values('Año')  # Empieza en 1950
df_viej_3 = df3_long.copy().sort_values('Año')     # Termina en 1949

# A. Valor 1950 (base nueva)
val_1950_nuevo = df_ref_4.loc[df_ref_4['Año'] == 1950, 'Valor'].values[0]

# B. Tasa de crecimiento manual (PIB agregado)
pib_1949 = 51608.13
pib_1950 = 52117.61

gt_pib = (pib_1950 / pib_1949) - 1

# C. Ajuste del nivel de 1949 consistente con 1950
val_1949_ajustado = val_1950_nuevo / (1 + gt_pib)

# D. Factor de escala
v_1949_viej = df_viej_3.loc[df_viej_3['Año'] == 1949, 'Valor'].values[0]
factor_final_df3 = val_1949_ajustado / v_1949_viej

# E. Aplicar ajuste a toda la serie 1925–1949
df_viej_3['Valor'] = df_viej_3['Valor'] * factor_final_df3
series_ajustadas[3] = df_viej_3

print(f"Empalme DF3 con PIB manual (gt={gt_pib:.6f})")

# --- 4. UNIÓN FINAL ---
panel_final = []
# Unimos desde 8 hasta 3
for i in range(8, 2, -1):
    df_temp = series_ajustadas[i]
    if not panel_final:
        panel_final.append(df_temp)
    else:
        # Solo años que no estén en la serie superior
        anio_min_superior = pd.concat(panel_final)['Año'].min()
        panel_final.append(df_temp[df_temp['Año'] < anio_min_superior])

df_100_final = pd.concat(panel_final).sort_values('Año').reset_index(drop=True)


Empalme DF3 con PIB manual (gt=0.009872)


### Nota metodológica: Empalme y ajuste de la serie histórica (1925–1950)

La construcción de la serie de tiempo del producto agropecuario y del consumo de los hogares para el periodo 1925–2025 requirió la implementación de un procedimiento de empalme debido a la coexistencia de diferentes años base en las cuentas nacionales y a la ausencia de información homogénea para los primeros años del periodo de estudio.

Para el periodo 1950–2025, la serie fue construida mediante un empalme por encadenamiento entre seis segmentos correspondientes a distintos años base, expresando finalmente todas las variables en términos de la base 2015. Este procedimiento permite preservar la dinámica de las tasas de crecimiento y garantizar la comparabilidad intertemporal de la serie.

Sin embargo, el empalme del periodo 1925–1949 presentó una dificultad adicional. Las series del producto agropecuario y del consumo de los hogares disponibles para este periodo fueron construidas retrospectivamente con base en el año 1950, lo que generó inconsistencias en niveles, particularmente la presencia de valores superiores a los del año base, lo cual resulta incompatible desde el punto de vista contable.

Adicionalmente, la presencia de un shock histórico significativo asociado a los eventos de 1948 (El Bogotazo) introduce una distorsión en las tasas de crecimiento entre 1948 y 1949, reflejando un efecto de recuperación transitoria más que una dinámica estructural de la economía. En consecuencia, el uso directo de dichas tasas podría sesgar la trayectoria de la serie.

Para corregir estas inconsistencias, se utilizó como referencia la tasa de crecimiento del PIB agregado proveniente de la serie histórica del Banco de la República (1905–2025), la cual incorpora ajustes metodológicos y garantiza consistencia intertemporal. Esta tasa fue empleada como una aproximación para reescalar la dinámica del producto agropecuario y del consumo de los hogares en el periodo 1925–1949.

El procedimiento consiste en ajustar los niveles de las series mediante la aplicación iterativa de las tasas de crecimiento del PIB agregado, partiendo del nivel observado en el año base (1950), de acuerdo con la siguiente expresión:

$$
Y_{t-1} = \frac{Y_t}{1 + g_t}
$$

donde $Y_t$ corresponde al nivel de la variable en el año $t$, y $g_t$ es la tasa de crecimiento del PIB agregado entre $t-1$ y $t$.

Este enfoque implica asumir que, en ausencia de información sectorial consistente, la dinámica del PIB agregado constituye una aproximación razonable para la evolución de las variables consideradas. Si bien este supuesto introduce una simplificación, permite garantizar la coherencia intertemporal de la serie y evitar distorsiones derivadas de shocks transitorios o inconsistencias en los niveles.

En conjunto, la combinación de empalme por encadenamiento y ajuste mediante tasas de crecimiento agregadas permite construir una serie histórica larga, consistente y adecuada para el análisis econométrico de largo plazo.

# Descargar la base

Ahora vamos a descargar la base de datos que acabo de construir para trabajar todo el proceso de econometria en R.

In [12]:
# Vamos a descargar la base de datos final para trabajar en R
df_100_final.to_csv('Base_100_Anios_Construccion.csv', index=False)

# PIB por Demanda | Consumo

En este apartado haremos exactamente lo mismo que la metodología anterior pero para el PIB en su apartado de consumo, como estas series fueron construidas por el Banco de la República con la misma metodología la metodología de enpalme es la misma

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# para leer los datos en xlsx
import seaborn as sns

In [14]:
# vamos a leer todas las bases de datos, recordando que esta vez van a empezar desde File (9).xlsx hasta el 14
for i in range(9, 15):
    # Creamos la variable globalmente con el nombre dinámico
    globals()[f'df{i}'] = pd.read_excel(f'File ({i}).xlsx', sheet_name='Series de datos', skiprows=1)
    print(f'Variable df{i} creada exitosamente.')

sectores = ["1.03.01.1. Consumo total - hogares", 
"1.02.02.1.1. Consumo final - gastos de consumo\nprivado",
"1.02.01.1.1. Consumo final - hogares",
"1.01.01.1. Consumo de Hogares",
"1.01.01.1. Consumo de Hogares",
"1.02.01. Gasto de consumo final individual de los\nhogares; gasto de consumo final de las ISFLH"]

for i in range(9, 15):
    # 1. Traemos el dataframe original usando su nombre de variable
    df_actual = globals()[f'df{i}']
    
    # 2. Limpiamos y filtramos el dataframe
    df_actual.dropna(inplace=True)
    filtro_agri = df_actual[df_actual['Unnamed: 0'].isin(sectores)].copy()
    
    # 3. Creamos la nueva variable 
    globals()[f'df{i}_agri'] = filtro_agri
    
    print(f'Creada exitosamente la variable: df{i}_agri con {len(filtro_agri)} filas.')



Variable df9 creada exitosamente.
Variable df10 creada exitosamente.
Variable df11 creada exitosamente.
Variable df12 creada exitosamente.
Variable df13 creada exitosamente.
Variable df14 creada exitosamente.
Creada exitosamente la variable: df9_agri con 1 filas.
Creada exitosamente la variable: df10_agri con 1 filas.
Creada exitosamente la variable: df11_agri con 1 filas.
Creada exitosamente la variable: df12_agri con 1 filas.
Creada exitosamente la variable: df13_agri con 1 filas.
Creada exitosamente la variable: df14_agri con 1 filas.


como hicimos antes, vamos a cambiar de formato largo a ancho

In [15]:
# --- PROCESAMIENTO HOMOLOGADO PARA CONSUMO (FILES 9–14) ---

sectores_interes = [
    "1.03.01.1. Consumo total - hogares", 
    "1.02.02.1.1. Consumo final - gastos de consumo\nprivado",
    "1.02.01.1.1. Consumo final - hogares",
    "1.01.01.1. Consumo de Hogares",
    "1.02.01. Gasto de consumo final individual de los\nhogares; gasto de consumo final de las ISFLH"
]

# --- 1. CARGA ---
for i in range(9, 15):
    globals()[f'df{i}'] = pd.read_excel(
        f'File ({i}).xlsx', 
        sheet_name='Series de datos', 
        skiprows=1
    )
    print(f'df{i} cargado.')

# --- 2. LIMPIEZA + HOMOLOGACIÓN + LONG ---
for i in range(9, 15):
    df_actual = globals()[f'df{i}'].copy()
    
    # Limpiar
    df_actual.dropna(subset=['Unnamed: 0'], inplace=True)
    
    # Filtrar consumo
    df_filtrado = df_actual[df_actual['Unnamed: 0'].isin(sectores_interes)].copy()
    
    # Homologar nombre
    df_filtrado['Unnamed: 0'] = 'Consumo Hogares'
    
    # Melt
    df_long = pd.melt(
        df_filtrado,
        id_vars=['Unnamed: 0'],
        var_name='Fecha',
        value_name='Valor'
    )
    
    # Limpieza final
    df_long.rename(columns={'Unnamed: 0': 'Sector'}, inplace=True)
    df_long['Fecha'] = pd.to_datetime(df_long['Fecha'], dayfirst=True, errors='coerce')
    df_long['Año'] = df_long['Fecha'].dt.year
    
    # Guardar
    globals()[f'df{i}_long'] = df_long.sort_values('Año')
    
    print(f'df{i}_long creado (Consumo Hogares).')

df9 cargado.
df10 cargado.
df11 cargado.
df12 cargado.
df13 cargado.
df14 cargado.
df9_long creado (Consumo Hogares).
df10_long creado (Consumo Hogares).
df11_long creado (Consumo Hogares).
df12_long creado (Consumo Hogares).
df13_long creado (Consumo Hogares).
df14_long creado (Consumo Hogares).


Con el formato long de cada serie vamos a empalmar las series

In [16]:
# primero miramos las fechas para ver si hay traslapes o no entre las series
for i in range(9, 15):
    df_actual = globals()[f'df{i}_long']
    print(f'Para df{i}_long: Fecha mínima = {df_actual["Fecha"].min()} | Fecha máxima = {df_actual["Fecha"].max()}')

Para df9_long: Fecha mínima = 1925-12-31 00:00:00 | Fecha máxima = 1949-12-31 00:00:00
Para df10_long: Fecha mínima = 1950-12-31 00:00:00 | Fecha máxima = 1970-12-31 00:00:00
Para df11_long: Fecha mínima = 1970-12-31 00:00:00 | Fecha máxima = 1996-12-31 00:00:00
Para df12_long: Fecha mínima = 1994-03-31 00:00:00 | Fecha máxima = 2007-12-31 00:00:00
Para df13_long: Fecha mínima = 2000-03-31 00:00:00 | Fecha máxima = 2017-12-31 00:00:00
Para df14_long: Fecha mínima = 2005-03-31 00:00:00 | Fecha máxima = 2025-12-31 00:00:00


Vemos como tiene el mismo problema de los primeros df, pero antes vamos a pasar de trimestral a anual el df12 al 14

In [17]:
# trimestral a anual
dfs_trimestrales = [12, 13, 14]
for i in dfs_trimestrales:
    df_temp = globals()[f'df{i}_long'].copy()
    
    # 1. Ponemos la fecha como índice (necesario para resample)
    df_temp.set_index('Fecha', inplace=True)
    
    # 2. Agrupamos por año ('YE' o 'Y') y sumamos los valores
    df_anual = df_temp.resample('YE').sum(numeric_only=True).reset_index()
    
    # 3. Re-agregamos la columna de Sector (que se pierde al sumar)
    df_anual['Sector'] = 'Consumo Hogares'
    
    # 4. Guardamos la nueva versión anual
    globals()[f'df{i}_anual'] = df_anual.sort_values('Año')
    
    print(f'df{i} convertido a anual. Periodos: {len(df_anual)}')

df12 convertido a anual. Periodos: 14
df13 convertido a anual. Periodos: 18
df14 convertido a anual. Periodos: 21


Repetimos el mismo codigo, repiendo lo de arriba, por si solo se quiere correr este 

In [18]:
# --- 1. PROCESAMIENTO INICIAL (CONSUMO HOGARES) ---
sectores_interes = [
    "1.03.01.1. Consumo total - hogares", 
    "1.02.02.1.1. Consumo final - gastos de consumo\nprivado",
    "1.02.01.1.1. Consumo final - hogares",
    "1.01.01.1. Consumo de Hogares",
    "1.02.01. Gasto de consumo final individual de los\nhogares; gasto de consumo final de las ISFLH"
]

for i in range(9, 15):
    df = pd.read_excel(f'File ({i}).xlsx', sheet_name='Series de datos', skiprows=1)
    df.dropna(subset=['Unnamed: 0'], inplace=True)
    df_f = df[df['Unnamed: 0'].isin(sectores_interes)].copy()
    df_f['Unnamed: 0'] = 'Consumo Hogares'
    
    # Melt y Fechas
    df_l = pd.melt(df_f, id_vars=['Unnamed: 0'], var_name='Fecha', value_name='Valor')
    df_l.rename(columns={'Unnamed: 0': 'Sector'}, inplace=True)
    df_l['Fecha'] = pd.to_datetime(df_l['Fecha'], dayfirst=True, errors='coerce')
    df_l['Año'] = df_l['Fecha'].dt.year
    
    if i in [9, 10, 11]:
        globals()[f'df{i}_long'] = df_l.sort_values('Año')
    else:
        # Anualización para 12, 13 y 14
        df_a = df_l.resample('YE', on='Fecha').sum(numeric_only=True).reset_index()
        df_a['Año'] = df_a['Fecha'].dt.year
        df_a['Sector'] = 'Consumo Hogares'
        globals()[f'df{i}_anual'] = df_a.sort_values('Año')

# --- 2. EMPALME EN CASCADA ---
series_ajustadas = {14: df14_anual.copy()}
nombres_map = {
    13: 'df13_anual', 
    12: 'df12_anual', 
    11: 'df11_long', 
    10: 'df10_long', 
    9:  'df9_long'
}

for i in range(13, 9, -1):
    df_ref = series_ajustadas[i+1]
    df_viej = globals()[nombres_map[i]].copy()
    
    comunes = set(df_ref['Año']) & set(df_viej['Año'])
    anio_pivote = max(comunes)
    
    factor = df_ref.loc[df_ref['Año'] == anio_pivote, 'Valor'].values[0] / \
             df_viej.loc[df_viej['Año'] == anio_pivote, 'Valor'].values[0]
    
    df_viej['Valor'] = df_viej['Valor'] * factor
    series_ajustadas[i] = df_viej

# --- 3. EMPALME df9 -> df10 USANDO PIB MANUAL (1949-1950) ---

df_ref_10 = series_ajustadas[10].sort_values('Año')
df_viej_9 = df9_long.copy().sort_values('Año')

# A. Valor del primer año de la serie nueva (1950)
val_1950_nuevo = df_ref_10.loc[
    df_ref_10['Año'] == df_ref_10['Año'].min(), 'Valor'
].values[0]

# B. Tasa de crecimiento manual PIB agregado
pib_1949 = 51608.13
pib_1950 = 52117.61
gt_pib = (pib_1950 / pib_1949) - 1

# C. Ajuste del último valor (1949)
v_last = df_viej_9.iloc[-1]['Valor']  # 1949

val_last_ajustado = val_1950_nuevo / (1 + gt_pib)
factor_final_df9 = val_last_ajustado / v_last

# D. Aplicar ajuste
df_viej_9['Valor'] = df_viej_9['Valor'] * factor_final_df9
series_ajustadas[9] = df_viej_9

print(f"Empalme DF9 con PIB manual (gt={gt_pib:.6f})")

# --- 4. UNIÓN FINAL ---
panel_final = []

for i in range(14, 8, -1):
    df_temp = series_ajustadas[i]
    if not panel_final:
        panel_final.append(df_temp)
    else:
        anio_min_superior = pd.concat(panel_final)['Año'].min()
        panel_final.append(df_temp[df_temp['Año'] < anio_min_superior])

df_consumo_final = pd.concat(panel_final).sort_values('Año').reset_index(drop=True)

Empalme DF9 con PIB manual (gt=0.009872)


In [19]:
# 4. Chequeo final: continuidad en el empalme
print(
    df_viej_9[df_viej_9['Año']==df_viej_9['Año'].max()]['Valor'].values[0],
    df_ref_10[df_ref_10['Año']==df_ref_10['Año'].min()]['Valor'].values[0]
)

40312.42388627082 40710.391681685556


listo, ahora descargamos el df de consumo final

In [20]:
df_consumo_final.to_csv('Base_100_Anios_Consumo.csv', index=False)